In [1]:
import pandas as pd


In [2]:
df = pd.read_csv(r"D:\Jupiter\Employee_project\data\emp_clean.csv", encoding="UTF-8")


# HR Employee Data Analysis Questions

*Basic Analysis*

In [ ]:
print(df["employee_id"].count())
print(df["employee_id"].nunique())
print(df["age"].mean())
print(df["age"].max())
print(df["age"].min())
print(df["salary"].mean())
print(df["salary"].max())
print(df["salary"].min())
print(df["experience_years"].mean())
print(df["training_hours"].mean())
print(df["gender"].value_counts())
print(df["department"].value_counts())
print(df["city"].value_counts())
print(df["education"].value_counts())
print(df["job_role"].value_counts())
print(df["promotion_status"].value_counts())
print(df["attrition_status"].value_counts())


*Pure Pandas DataFrame ke scientific notation ko disable karo*

In [4]:
pd.set_option('display.float_format', '{:,.2f}'.format)

*GroupBy Analysis*

In [ ]:
department_summary = df.groupby("department").agg(
    avg_slary = ("salary", "mean"),
    avg_age = ("age", "mean"),
    avg_exp = ("experience_years", "mean"),
    avg_training = ("training_hours", "mean"),
    promotion_count = ("promotion_status", "count"),
    attration_count = ("attrition_status", "count")
)

gender_summary = df.groupby("gender").agg(
    avg_Salary = ("salary", "mean"),
    avg_exp = ("experience_years", "mean"),
    total_promotion = ("promotion_status", "count")
   
)

# print(gender_summary)

gender_promotion_rate = (
    df.groupby("gender")["promotion_status"]
      .apply(lambda x: (x == "Yes").mean() * 100)
      .round(2)
)

print(gender_promotion_rate)

*Department*

In [6]:
dept_no = df.groupby("department")["employee_id"].count()
dept_expd = df.groupby("department")["salary"].sum()
avg_salary_dept = df.groupby("department")["salary"].mean()
avg_emp_age = df.groupby("department")["age"].mean()
avg_exp = df.groupby("department")["experience_years"].mean()
dept_attrn = df.groupby("department")["attrition_status"].apply(lambda x:(x == "Yes").mean() * 100).round()
high_dept_emp = df.groupby("department")["employee_id"].count().sort_values(ascending=False).head(1)
high_salary_expd = df.groupby("department")["salary"].sum().sort_values(ascending=False).head(1)
high_attrn = df.groupby("department")["attrition_status"].apply(lambda x: (x== "Yes").sum()).sort_values(ascending=False).head(1)

*Job Role Analysis*

In [7]:
job_role_analysis = df.groupby("job_role").agg(
    no_of_emp = ("employee_id", "count"),
    avg_salary = ("salary", "mean"),
    avg_exp = ("experience_years", "mean"),
    avg_age = ("age", "mean")
    )



attrn_rate = df[df["attrition_status"] == "Yes"].groupby("job_role").size()
total_emp = df.groupby("job_role").size()
job_role_attrn_rate = ((attrn_rate / total_emp) * 100)


high_salary_job_role = df.groupby("job_role")["salary"].max().sort_values(ascending=False).head(1)


high_attrn_rate = df[df["attrition_status"] == "Yes"].groupby("job_role").size()
total_emp = df.groupby("job_role").size()
high_attrn_rate = ((attrn_rate / total_emp) * 100).sort_values(ascending=False).head(1)




*Salary Analysis*

In [ ]:
top_10_high_paid = df.nlargest(10, "salary")

print(top_10_high_paid[["employee_id", "name", "job_role", "department", "salary"]])

avg_salary_gender = df.groupby(["department","gender"])["salary"].mean()
print(avg_salary_gender)

company_Avg_salary = df["salary"].mean()
print(df.loc[company_Avg_salary < df["salary"]], ("name", "salary")) 

print(df.loc[(df["experience_years"] >= 7) & (df["salary"] < company_Avg_salary)],( "name", "salary", "experience_years"))


largest_salary_variation = (
    df.groupby("department")["salary"]
    .agg(lambda x: x.max() - x.min())
    .sort_values(ascending=False)
    .head(1)
)
print(largest_salary_variation)

*Experience Analysis*

In [9]:
df["experience_level"] = df["experience_years"].apply(lambda x: "Senior" if x >= 8
                                                else "Mid level" if x >= 6
                                                else "Junior" if x >= 3
                                                else "Fresher")
print(df.loc[df["experience_years"].value_counts(),("name","experience_level")])

avg_salary_exp = df.groupby("experience_level")["salary"].mean()
attrn_rate_exp = df.groupby("experience_level")["attrition_status"].apply(lambda x:( x == "Yes").mean() * 100)

avg_salary = df["salary"].mean()
high_exp_low_salary = df[(df["experience_level"] == "Senior") & (df["salary"] < avg_salary)]

order = ["Fresher", "Junior", "Mid level", "Senior"]
avg_sal_exp = df.groupby("experience_level")["salary"].mean().reindex(order)


             name experience_level
109   Rohan Mehta        Mid level
51    Akash Patel        Mid level
48   Tanya Kapoor           Senior
28   Aarav Sharma           Junior
22   Aditya-Singh          Fresher
..            ...              ...
1      Amit Kumar           Senior
1      Amit Kumar           Senior
1      Amit Kumar           Senior
1      Amit Kumar           Senior
1      Amit Kumar           Senior

[163 rows x 2 columns]


*Gender Analysis*

In [10]:
gender_dist = df.groupby("gender")["employee_id"].count()
emp_count = df.groupby(["gender", "department"])["employee_id"].count()
gender_avg_salary = df.groupby("gender")["salary"].mean()
attrn_rate_gender = df.groupby("gender")["attrition_status"].apply(lambda x: (x == "Yes").mean() * 100)
dept_gender_count = df.groupby("department")["gender"].value_counts()



*Attrition Analysis*

In [11]:
total_left_emp = df[["attrition_status"]].apply(lambda x : (x == "Yes")).sum()

totl_emp =  df["employee_id"].count()
overall_attrn = (total_left_emp / totl_emp) * 100


df["attrition_numeric"] = df["attrition_status"].apply(
    lambda x: 1 if x == "Yes" else 0
)

attrn_dept = df.groupby("department")["attrition_numeric"].agg("mean") * 100
attrn_job_role = df.groupby("job_role")["attrition_numeric"].agg("mean") * 100
attrn_gender = df.groupby("gender")["attrition_numeric"].agg("mean") * 100

df["group_age"] = df["age"].apply(
    lambda x: "Young" if x <= 25
    else "Adult" if x <= 35
    else "Middle Age" if x <= 45
    else "Senior"
)

attrn_age_group = df.groupby("group_age")["attrition_numeric"].agg("mean") * 100

attrn_exp_level = df.groupby("experience_level")["attrition_numeric"].agg("mean") * 100

highest_attrn = {
    "Department": attrn_dept.max(),
    "Job Role": attrn_job_role.max(),
    "Gender": attrn_gender.max(),
    "Age Group": attrn_age_group.max(),
    "Experience Level": attrn_exp_level.max()
}

print(highest_attrn)
print(max(highest_attrn, key=highest_attrn.get))

{'Department': 25.842696629213485, 'Job Role': 41.66666666666667, 'Gender': 23.643410852713178, 'Age Group': 22.758620689655174, 'Experience Level': 22.636103151862464}
Job Role


*Performance Analysis*

In [12]:
avg_perform = df["performance_rating"].mean()
avg_dept_perform = df.groupby("department")["performance_rating"].mean()
avg_salary_perform = df.groupby("performance_rating")["salary"].mean()
avg_exp_perform = df.groupby("performance_rating")["experience_years"].mean()

high_perform_sal_low = df[(df["performance_rating"] > 8) & (df["salary"] < avg_salary)]
high_dept_perform = df.groupby("department")["performance_rating"].mean().idxmax()

*Promotion Analysis*

In [ ]:

dept_promoted = (df[df["promotion_status"] == "Yes"].groupby("department").size())
total_dept_emp = df.groupby("department").size()
dept_promotion_Rate = (dept_promoted / total_dept_emp)  * 100



gender_promoted = (df[df["promotion_status"] == "Yes"].groupby("gender").size())
total_gender_emp = df.groupby("gender").size()
gender_promotion_Rate = (gender_promoted / total_gender_emp) * 100


avg_promoted_exp = df[df["promotion_status"] == "Yes"]["experience_years"].mean()


high_exp_no_promoted = df[(df["experience_years"] > 8) & (df["promotion_status"] == "No")]


high_perform_no_promoted = df[(df["performance_rating"] > 8) & (df["promotion_status"] == "No")]
